# Timeseries

## Imports

In [ ]:
import src.utils
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import seaborn as sns
import sklearn.linear_model
import pathlib
import os
import sklearn.linear_model
import cmocean
import pandas as pd
import copy
import scipy.stats
import matplotlib as mpl
import matplotlib.patheffects as pe

## RNG
rng = np.random.default_rng()

## color palette
sns.set(rc={"axes.facecolor": "white", "axes.grid": False}, palette="colorblind")

## paths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
# SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## funcs

In [ ]:
def get_nino3(x, z_t=70, lat_bound=5):
    """get Niño 3 average, with checks in place"""

    if "z_t" in x.dims:
        x = x.sel(z_t=z_t, method="nearest")

    if "latitude" in x.dims:
        x = x.sel(latitude=slice(-lat_bound, lat_bound)).mean("latitude")

    return x.sel(longitude=slice(210, 270)).mean("longitude")


def get_nino3_helper(x, lat_bounds):
    """get Niño 3 average, with checks in place"""

    ## get index to select data
    idx = dict(latitude=slice(*lat_bounds), longitude=slice(210, 270))

    return x.sel(idx).mean(["longitude", "latitude"])


def get_w_sub(w, mld=50, delta=30):
    """get velocity at base of mixed layer"""

    ## get temperature difference
    w_sub = w.sel(z_t=slice(mld, mld + delta)).mean("z_t")

    return w_sub


def get_dTdz_sub(Tsub, mld=50, delta=30):
    """get difference between surface and subsurface temp"""

    ## get temperature difference
    dT = src.utils.get_dT_sub(Tsub=Tsub, Hm=mld, delta=delta)

    ## get gradient
    dTdz = -dT / mld

    return dTdz


def get_dTdz_sub2(Tsub, mld=80):
    """get difference between surface and subsurface temp"""

    ## get temperature difference
    dT = Tsub.sel(z_t=mld, method="nearest") - Tsub.sel(z_t=0, method="nearest")

    ## get gradient
    dTdz = -dT / mld

    return dTdz

### individual metrics

#### Niño 3 averages

In [ ]:
## load spatial data
forced, anom_ = src.utils.load_consolidated()

## get subset of data for total
VARNAMES = ["sst"]
total = anom_[VARNAMES] + forced[VARNAMES]
total = xr.merge([forced[[f"{v}_comp" for v in VARNAMES]], total])

#### $\partial T / \partial x$

In [ ]:
## compute diff. def of Tw
sel_lat = lambda x: x.sel(latitude=slice(-5, 5)).mean(["latitude", "longitude"])
get_Tw = lambda x: sel_lat(x.sel(longitude=slice(150, 190)))
get_Te = lambda x: sel_lat(x.sel(longitude=slice(240, 280)))
get_dTdx = lambda x: get_Tw(x) - get_Te(x)

## compute dTdx
dTdx = src.utils.reconstruct_wrapper(total[["sst", "sst_comp"]], get_dTdx)
dTdx = dTdx.rename({"sst": "dTdx"})

## same, but anomalies
dTdxa = src.utils.reconstruct_wrapper(anom_[["sst", "sst_comp"]], get_dTdx)
dTdxa = dTdxa.rename({"sst": "dTdxa"})

## merge
dTdx = xr.merge([dTdx, dTdxa])

#### $T$ indices

In [ ]:
# ## load tropical SST avg
trop_sst = xr.open_dataset(pathlib.Path(DATA_FP, "cesm/trop_sst.nc"))

## Load T,h (total)
Th_total = xr.open_dataset(DATA_FP / "cesm" / "Th.nc")
Th_total = xr.merge([Th_total, trop_sst])
Th_anom = Th_total - Th_total.mean("member")

## compute relative sst
for n in ["T_3", "T_34", "T_4"]:
    Th_total[f"{n}_rel"] = Th_total[n] - trop_sst["trop_sst_10"]

### Combine data

In [ ]:
## specify time index for plotting
## start index: use 3 for SON, 4 for DJF, etc.
time_idx = slice(0, None, None)
# time_idx = slice(0,None,4) ## for JFM

## combine data
data = xr.merge([Th_total, dTdx, Th_anom["T_34"].rename("T_34a")])

## get windowed version
data = src.utils.get_windowed(
    data.isel(time=slice(None, -1)), window_size=480, stride=60
)

## resample to DJF
data = data.resample({"time": "QS-DEC"}).mean().isel(time=time_idx)

## compute ensemble mean and sigma
data_mean = data.mean("time")
data_sigma = data.std("time")

### Remove nonlinear rect. effect

In [ ]:
coefs = src.utils.estimate_rect(
    xr.merge([data_sigma["T_34a"].rename("sigma_T"), data_mean]),
    xvar="sigma_T",
    yvars=list(data_mean),
)

data_norect = src.utils.remove_rect(
    xr.merge([data_sigma["T_34a"].rename("sigma_T"), data_mean]),
    xvar="sigma_T",
    yvars=list(data_mean),
)

### Plot first method of correcting gradient

In [ ]:
## specify year index
years = [1870, 2010, 2080]
colors = sns.color_palette("colorblind")

fig, ax = plt.subplots(figsize=(5, 4), layout="constrained")
for y, c in zip(years, colors):

    ## get data and compute correlation
    xx = data_sigma["T_34a"].sel(year=y)
    yy = data_mean["dTdx"].sel(year=y)

    r = xr.corr(xx, yy).values.item()

    ## scatter data
    ax.scatter(xx, yy, s=15, color=c, label=f"{y}, $r=${r:.2f}")

    ## plot best fit
    zz = np.linspace(0, 1.6)
    coef_y = coefs["dTdx"].sel(year=y)
    m = coef_y.sel(degree=1).values.item()
    b = coef_y.sel(degree=0).values.item()
    ax.plot(zz, zz * m + b, c=c)

    ## highlight intercept and mean
    ax.scatter(0, b, color=c, marker="*", s=100, zorder=10, edgecolor="k")
    ax.scatter(
        xx.mean(), yy.mean(), color=c, marker="s", s=50, zorder=10, edgecolor="k"
    )

ax.legend()
ax.axvline(0, ls="--", c="k", lw=1)
ax.set_ylim([None, 5.5])
ax.set_ylabel(r"$\overline{T_w}-\overline{T_e}$ (K)")
ax.set_xlabel(r"$\sigma(T_{34})$")
ax.set_yticks([3, 4, 5])
ax.set_xticks([0, 1, 1.5])
plt.show()

### scatter plot of nonlinear relationships over time

In [ ]:
def psi_(x, a, b, c, e):
    """base transfer function"""
    return e * x**3 + c * x**2 + b * x + a
    # return c * x**2 + b * x + a


def get_psi(x, y):
    """get transfer function from data"""

    ## prep data
    stack = lambda x: x.stack(s=["time", "member"]).values
    x_ = stack(x)
    y_ = stack(y)

    ## get parameter fit
    p, _ = scipy.optimize.curve_fit(
        f=psi_,
        xdata=x_,
        ydata=y_,
    )

    ## define transfer func
    psi = lambda x: psi_(x, a=p[0], b=p[1], c=p[2], e=p[3])

    return psi

Simpler: plot intercept for fit over time?

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(10, 4), layout="constrained")

psi_ys = []

for ax, y, c in zip(axs, years, colors):
    data_y = data.sel(year=y).isel(time=slice(0, None, 4))

    ## plot data
    ax.scatter(data_y["T_34a"], data_y["dTdx"], s=5, color=c, alpha=0.5)

    ## plot mean
    ax.scatter(
        data_y["T_34a"].mean(),
        data_y["dTdx"].mean(),
        s=50,
        marker="s",
        color=c,
        zorder=10,
        edgecolor="k",
    )

    ax_kwargs = dict(ls="--", c="k", lw=0.8)
    ax.axhline(0, **ax_kwargs)
    ax.axvline(0, **ax_kwargs)
    ax.set_xlabel(r"$T_{34}$")

    ## plot best fit
    psi_ys.append(get_psi(x=data_y["T_34a"], y=data_y["dTdx"]))

    ## plot best fits
    fit_kwargs = dict(
        lw=2.5,
        ls="-",
        path_effects=[pe.Stroke(linewidth=4, foreground="w"), pe.Normal()],
    )
    zz = np.linspace(-4, 3)
    ax.plot(zz, psi_ys[-1](zz), c=c, **fit_kwargs)

axs[1].plot(zz, psi_ys[0](zz), c=colors[0], **fit_kwargs)
axs[2].plot(zz, psi_ys[0](zz), c=colors[0], **fit_kwargs)
axs[2].plot(zz, psi_ys[1](zz), c=colors[1], **fit_kwargs)

src.utils.set_lims(axs)
axs[0].set_yticks([-2, 0, 2, 4])
axs[0].set_ylabel(r"$T_w-T_e$")
for ax in axs[1:]:
    ax.set_yticks([])
plt.show()

In [ ]:
intercepts = []

for y in data.year.values[:-1]:

    ## get data and fit
    data_y = data.sel(year=y).isel(time=slice(0, None, 4))
    psi_y = get_psi(x=data_y["T_34a"], y=data_y["dTdx"])

    ## get intercept
    intercepts.append(psi_y(data_y["T_34a"].median()))

intercepts = np.array(intercepts)

## Timeseries

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3.5))

## plot each quantile
for q, lw in zip([0.5, 0.05, 0.95], [2, 1, 1]):

    for data_, label, ls in zip(
        [data.mean("time"), data_norect], ["Raw", "Corrected"], ["-", "--"]
    ):

        ## get data
        data_q = data_["dTdx"].quantile(q=q, dim="member")

        ## plot label
        plot_label = label if (q == 0.5) else None

        ## plot
        ax.plot(data.year, data_q, c="gray", lw=lw, label=plot_label, ls=ls)

        ## plot start line
        if q == 0.5:
            ax.axhline(data_q.isel(year=0), lw=0.8, ls="--", c="k")

# ## plot intercepts
# ax.plot(data.year[:-1], intercepts+.45, c="k")
# ax.plot(data.year[:-1], intercepts+1.35, c="k")

## scatter points
for y, c in zip(years, colors):
    kwargs = dict(zorder=10, color=c, edgecolor="k")
    fn = lambda x: x["dTdx"].sel(year=y).mean("member")
    ax.scatter(y, fn(data.mean("time")), s=75, marker="s", **kwargs)
    ax.scatter(y, fn(data_norect), s=150, marker="*", **kwargs)

src.utils.add_vticks([ax], xticks=[1870, 2010, 2080], xlines=[2010])
ax.set_yticks([3, 4, 5])
ax.set_ylabel(r"$\overline{T_w}-\overline{T_e}$ (K)")
ax.legend()

plt.show()